In [ ]:
!pip install wikipedia-api -q
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent="RAGInternship/1.0 (anushreek8281@gmail.com)",
    language="en"
)

topics = ["Machine learning", "Neural network", "Deep learning", "Transformer model", "Natural language processing"]

for topic in topics:
    page = wiki.page(topic)
    with open(f"{topic.replace(' ', '_')}.txt", "w") as f:
        f.write(page.text)
    print(f"Saved: {topic}")

Saved: Machine learning
Saved: Neural network
Saved: Deep learning
Saved: Transformer model
Saved: Natural language processing


In [ ]:
!pip install langchain chromadb pypdf langchain-google-genai langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 15.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
    GROQ_KEY = userdata.get("GROQ_KEY")

    if not os.environ["GOOGLE_API_KEY"]:
        raise ValueError("GEMINI_KEY is empty — check Colab Secrets")
    if not GROQ_KEY:
        raise ValueError("GROQ_KEY is empty — check Colab Secrets")

    print("✅ API keys loaded successfully")

except KeyError as e:
    print(f"❌ Secret not found in Colab Secrets: {e}")
    print("👉 Go to the 🔑 icon on the left sidebar and add your keys")
except ValueError as e:
    print(f"❌ Key error: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ API keys loaded successfully


In [ ]:
import os

for f in os.listdir("/content/"):
    print(f)

.config
Machine_learning.txt
Deep_learning.txt
Neural_network.txt
chroma_db
Natural_language_processing.txt
=0.2.0
Transformer_model.txt
sample_data


In [ ]:
from langchain_community.document_loaders import TextLoader
import os

try:
    all_docs = []
    failed_files = []

    filenames = [
        "Machine_learning.txt", "Deep_learning.txt", "Neural_network.txt",
        "Natural_language_processing.txt", "Transformer_model.txt"
    ]

    for filename in filenames:
        filepath = f"/content/{filename}"
        try:
            if not os.path.exists(filepath):
                print(f"⚠️ File not found: {filename} — skipping")
                failed_files.append(filename)
                continue

            loader = TextLoader(filepath)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"✅ Loaded: {filename} — {len(docs)} page(s)")

        except Exception as e:
            print(f"❌ Failed to load '{filename}': {e}")
            failed_files.append(filename)

    if not all_docs:
        raise ValueError("No documents loaded — check that your .txt files exist in /content/")

    print(f"\n📊 Total documents loaded: {len(all_docs)}")
    if failed_files:
        print(f"⚠️ Skipped files: {failed_files}")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Loaded: Machine_learning.txt — 1 page(s)
✅ Loaded: Deep_learning.txt — 1 page(s)
✅ Loaded: Neural_network.txt — 1 page(s)
✅ Loaded: Natural_language_processing.txt — 1 page(s)
✅ Loaded: Transformer_model.txt — 1 page(s)

📊 Total documents loaded: 5


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    if not all_docs:
        raise ValueError("all_docs is empty — run Cell 4 first")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(all_docs)

    if not chunks:
        raise ValueError("No chunks created — documents may be empty")

    print(f"✅ Total chunks created: {len(chunks)}")
    print(f"📄 Sample chunk:\n{chunks[0].page_content[:300]}...")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Total chunks created: 247
📄 Sample chunk:
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have all...


In [ ]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

try:
    if not chunks:
        raise ValueError("chunks is empty — run Cell 5 first")

    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    batch_size = 50
    vectorstore = None
    total_batches = (len(chunks) + batch_size - 1) // batch_size

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        batch_num = i // batch_size + 1

        try:
            print(f"⏳ Embedding batch {batch_num}/{total_batches} — chunks {i} to {i + len(batch)}...")

            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=batch,
                    embedding=embeddings,
                    persist_directory="./chroma_db"
                )
            else:
                vectorstore.add_documents(batch)

            print(f"✅ Batch {batch_num} stored successfully")

            if i + batch_size < len(chunks):
                print("⏸️ Waiting 30 seconds to avoid rate limit...")
                time.sleep(30)

        except Exception as e:
            print(f"❌ Batch {batch_num} failed: {e}")
            print("⏸️ Waiting 60 seconds before retrying...")
            time.sleep(60)
            try:
                if vectorstore is None:
                    vectorstore = Chroma.from_documents(
                        documents=batch,
                        embedding=embeddings,
                        persist_directory="./chroma_db"
                    )
                else:
                    vectorstore.add_documents(batch)
                print(f"✅ Batch {batch_num} retry successful")
            except Exception as retry_error:
                print(f"❌ Batch {batch_num} retry also failed: {retry_error}")
                print("⚠️ Skipping this batch and continuing...")

    if vectorstore is None:
        raise ValueError("No data was stored — all batches failed")

    vectorstore.persist()
    print(f"\n✅ Done! ChromaDB saved to ./chroma_db")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

⏳ Embedding batch 1/5 — chunks 0 to 50...
✅ Batch 1 stored successfully
⏸️ Waiting 30 seconds to avoid rate limit...
⏳ Embedding batch 2/5 — chunks 50 to 100...
✅ Batch 2 stored successfully
⏸️ Waiting 30 seconds to avoid rate limit...
⏳ Embedding batch 3/5 — chunks 100 to 150...
✅ Batch 3 stored successfully
⏸️ Waiting 30 seconds to avoid rate limit...
⏳ Embedding batch 4/5 — chunks 150 to 200...
✅ Batch 4 stored successfully
⏸️ Waiting 30 seconds to avoid rate limit...
⏳ Embedding batch 5/5 — chunks 200 to 247...
✅ Batch 5 stored successfully

✅ Done! ChromaDB saved to ./chroma_db


/tmp/ipykernel_4828/3883075276.py:58: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
import shutil

try:
    shutil.make_archive("/content/chroma_db_backup", "zip", "/content", "chroma_db")
    print("✅ chroma_db zipped — download it from the Files panel on the left")
except Exception as e:
    print(f"❌ Failed to zip: {e}")

✅ chroma_db zipped — download it from the Files panel on the left
